# FakeAVCeleb Dataset Audit Report

This notebook performs a comprehensive analysis of the FakeAVCeleb dataset.

**Date:** June 22, 2026
**Purpose:** Complete dataset inspection and audit without any preprocessing or model training.

## 1. Setup and Imports

In [1]:
import os
import pandas as pd
import numpy as np
import json
import cv2
import librosa
import soundfile as sf
from pathlib import Path
from collections import defaultdict, Counter
import random
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

# Set dataset path
DATASET_PATH = r"d:\uday\Vscode\Projects\Deepfake_Detection\Dataset_FakeAVCeleb"

print(f"Dataset Path: {DATASET_PATH}")
print(f"Path exists: {os.path.exists(DATASET_PATH)}")

Dataset Path: d:\uday\Vscode\Projects\Deepfake_Detection\Dataset_FakeAVCeleb
Path exists: True


## 2. Dataset Structure Analysis

In [2]:
# def get_directory_structure(path, max_depth=3, current_depth=0):
#     """
#     Recursively get directory structure up to max_depth
#     """
#     if current_depth > max_depth:
#         return {}
    
#     structure = {}
#     try:
#         items = sorted(os.listdir(path))
#         for item in items:
#             full_path = os.path.join(path, item)
#             if os.path.isdir(full_path):
#                 structure[item] = get_directory_structure(full_path, max_depth, current_depth + 1)
#             else:
#                 structure[item] = None
#     except PermissionError:
#         structure["ACCESS_DENIED"] = None
    
#     return structure

# # Get top-level structure
# print("=== TOP-LEVEL FOLDERS ===")
# top_level = sorted(os.listdir(DATASET_PATH))
# for folder in os.listdir(DATASET_PATH):
#     folder_path = os.path.join(DATASET_PATH, folder)
#     if os.path.isdir(folder_path):
#         item_count = len(os.listdir(folder_path))
#         print(f"📁 {folder}/ ({item_count} items)")
#     else:
#         size = os.path.getsize(folder_path)
#         print(f"📄 {folder} ({size:,} bytes)")

# print("\n=== COMPLETE FOLDER HIERARCHY (up to depth 3) ===")
# structure = get_directory_structure(DATASET_PATH, max_depth=3)


In [3]:
def count_files_by_extension(path):
    """
    Count files by extension recursively
    """
    extensions = defaultdict(int)
    total_files = 0
    total_size = 0
    
    for root, dirs, files in os.walk(path):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            extensions[ext] += 1
            total_files += 1
            try:
                total_size += os.path.getsize(os.path.join(root, file))
            except:
                pass
    
    return dict(extensions), total_files, total_size

print("=== FILE TYPE DISTRIBUTION ===")
extensions, total_files, total_size = count_files_by_extension(DATASET_PATH)

for ext, count in sorted(extensions.items(), key=lambda x: x[1], reverse=True):
    print(f"{ext}: {count:,} files")

print(f"\nTotal files: {total_files:,}")
print(f"Total size: {total_size / (1024**3):.2f} GB")
print(f"Average file size: {total_size / total_files / (1024**2):.2f} MB")

=== FILE TYPE DISTRIBUTION ===


.mp4: 21,544 files
.txt: 501 files
.csv: 1 files

Total files: 22,046
Total size: 6.16 GB
Average file size: 0.29 MB


## 3. Dataset Statistics

In [4]:
print("=== DETAILED FILE COUNTS BY TYPE ===")
video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv']
audio_extensions = ['.wav', '.mp3', '.flac', '.aac', '.ogg', '.m4a']
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif']

video_count = sum(extensions.get(ext, 0) for ext in video_extensions)
audio_count = sum(extensions.get(ext, 0) for ext in audio_extensions)
image_count = sum(extensions.get(ext, 0) for ext in image_extensions)

print(f"Video files: {video_count:,}")
print(f"Audio files: {audio_count:,}")
print(f"Image files: {image_count:,}")
print(f"Other files: {total_files - video_count - audio_count - image_count:,}")
print(f"\nTotal dataset size: {total_size / (1024**3):.2f} GB")
print(f"Average file size: {total_size / total_files / (1024**2):.2f} MB")

=== DETAILED FILE COUNTS BY TYPE ===
Video files: 21,544
Audio files: 0
Image files: 0
Other files: 502

Total dataset size: 6.16 GB
Average file size: 0.29 MB


## 4. Label Information Analysis

In [5]:
print("=== LABEL STORAGE ANALYSIS ===")

# Check for CSV files
csv_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

print(f"Found {len(csv_files)} CSV files:")
for csv_file in csv_files:
    print(f"  - {csv_file}")

# Check for JSON files
json_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith('.json'):
            json_files.append(os.path.join(root, file))

print(f"\nFound {len(json_files)} JSON files:")
for json_file in json_files:
    print(f"  - {json_file}")

# Check folder structure for labels
print("\n=== FOLDER-BASED LABELS ===")
print("Top-level folders that might indicate labels:")
for folder in os.listdir(DATASET_PATH):
    if os.path.isdir(os.path.join(DATASET_PATH, folder)):
        print(f"  - {folder}/")


=== LABEL STORAGE ANALYSIS ===
Found 1 CSV files:
  - d:\uday\Vscode\Projects\Deepfake_Detection\Dataset_FakeAVCeleb\meta_data.csv



Found 0 JSON files:

=== FOLDER-BASED LABELS ===
Top-level folders that might indicate labels:
  - FakeVideo-FakeAudio/
  - FakeVideo-RealAudio/
  - RealVideo-FakeAudio/
  - RealVideo-RealAudio/


In [6]:
# Read the metadata CSV file
metadata_path = os.path.join(DATASET_PATH, 'meta_data.csv')
if os.path.exists(metadata_path):
    print("=== METADATA CSV ANALYSIS ===")
    df_meta = pd.read_csv(metadata_path)
    print(f"Shape: {df_meta.shape}")
    print(f"\nColumns: {list(df_meta.columns)}")
    print(f"\nFirst 5 rows:")
    print(df_meta.head())
    print(f"\nData types:")
    print(df_meta.dtypes)
    print(f"\nBasic statistics:")
    print(df_meta.describe(include='all'))
else:
    print("No metadata CSV found")

=== METADATA CSV ANALYSIS ===
Shape: (21566, 10)

Columns: ['source', 'target1', 'target2', 'method', 'category', 'type', 'race', 'gender', 'path', 'Unnamed: 9']

First 5 rows:
    source target1 target2 method category                 type     race  \
0  id00076       -       -   real        A  RealVideo-RealAudio  African   
1  id00166       -       -   real        A  RealVideo-RealAudio  African   
2  id00173       -       -   real        A  RealVideo-RealAudio  African   
3  id00366       -       -   real        A  RealVideo-RealAudio  African   
4  id00391       -       -   real        A  RealVideo-RealAudio  African   

  gender       path                                         Unnamed: 9  
0    men  00109.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id...  
1    men  00010.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id...  
2    men  00118.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id...  
3    men  00118.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id...  
4

## 5. Class Distribution Analysis

In [7]:
print("=== CLASS DISTRIBUTION FROM FOLDER STRUCTURE ===")

# Analyze the four main categories
main_categories = [
    'FakeVideo-FakeAudio',
    'FakeVideo-RealAudio',
    'RealVideo-FakeAudio',
    'RealVideo-RealAudio'
]

class_stats = {}
for category in main_categories:
    category_path = os.path.join(DATASET_PATH, category)
    if os.path.exists(category_path):
        # Count files in this category
        file_count = 0
        for root, dirs, files in os.walk(category_path):
            file_count += len(files)
        class_stats[category] = file_count
        print(f"{category}: {file_count:,} files")

print("\n=== REAL VS FAKE SUMMARY ===")
real_video_real_audio = class_stats.get('RealVideo-RealAudio', 0)
fake_video_fake_audio = class_stats.get('FakeVideo-FakeAudio', 0)
fake_video_real_audio = class_stats.get('FakeVideo-RealAudio', 0)
real_video_fake_audio = class_stats.get('RealVideo-FakeAudio', 0)

total_real = real_video_real_audio + real_video_fake_audio
total_fake = fake_video_fake_audio + fake_video_real_audio
total_samples = total_real + total_fake

print(f"Total Real samples: {total_real:,}")
print(f"Total Fake samples: {total_fake:,}")
print(f"Real-to-Fake ratio: {total_real}:{total_fake} ({total_real/total_fake:.2f})")
print(f"Total samples: {total_samples:,}")

=== CLASS DISTRIBUTION FROM FOLDER STRUCTURE ===
FakeVideo-FakeAudio: 10,835 files
FakeVideo-RealAudio: 9,709 files
RealVideo-FakeAudio: 500 files
RealVideo-RealAudio: 1,000 files

=== REAL VS FAKE SUMMARY ===
Total Real samples: 1,500
Total Fake samples: 20,544
Real-to-Fake ratio: 1500:20544 (0.07)
Total samples: 22,044


## 6. Video Analysis

In [8]:
def analyze_video_file(video_path):
    """
    Analyze a single video file
    """
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return None
        
        # Get video properties
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = frame_count / fps if fps > 0 else 0
        
        cap.release()
        
        return {
            'duration': duration,
            'resolution': (width, height),
            'fps': fps,
            'frame_count': frame_count
        }
    except Exception as e:
        return None

def get_all_videos(path):
    """
    Get all video files recursively
    """
    video_files = []
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv']
    
    for root, dirs, files in os.walk(path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in video_extensions):
                video_files.append(os.path.join(root, file))
    
    return video_files

print("=== VIDEO ANALYSIS ===")
video_files = get_all_videos(DATASET_PATH)
print(f"Total video files found: {len(video_files):,}")

# Analyze a sample of videos (first 100 to avoid timeout)
sample_size = min(100, len(video_files))
sample_videos = random.sample(video_files, sample_size) if len(video_files) > 0 else []

video_stats = []
video_durations = []
resolutions = []
fps_list = []

for i, video_path in enumerate(sample_videos):
    if i % 10 == 0:
        print(f"Analyzing video {i+1}/{len(sample_videos)}...")
    
    stats = analyze_video_file(video_path)
    if stats:
        video_stats.append(stats)
        video_durations.append(stats['duration'])
        resolutions.append(stats['resolution'])
        fps_list.append(stats['fps'])

if video_stats:
    print(f"\nSuccessfully analyzed {len(video_stats)} videos")
    print(f"Average duration: {np.mean(video_durations):.2f} seconds")
    print(f"Min duration: {np.min(video_durations):.2f} seconds")
    print(f"Max duration: {np.max(video_durations):.2f} seconds")
    
    # Resolution analysis
    unique_resolutions = Counter(resolutions)
    print(f"\nUnique resolutions:")
    for res, count in unique_resolutions.most_common():
        print(f"  {res[0]}x{res[1]}: {count} videos")
    
    print(f"\nAverage resolution: {np.mean([r[0] for r in resolutions]):.0f}x{np.mean([r[1] for r in resolutions]):.0f}")
    
    # FPS analysis
    valid_fps = [f for f in fps_list if f > 0]
    if valid_fps:
        print(f"\nAverage FPS: {np.mean(valid_fps):.2f}")
        print(f"FPS range: {np.min(valid_fps):.2f} - {np.max(valid_fps):.2f}")
    
    # Video formats
    formats = Counter([os.path.splitext(v)[1].lower() for v in video_files])
    print(f"\nVideo formats:")
    for fmt, count in formats.most_common():
        print(f"  {fmt}: {count} videos")
else:
    print("No videos could be analyzed")

=== VIDEO ANALYSIS ===


Total video files found: 21,544
Analyzing video 1/100...


Analyzing video 11/100...


Analyzing video 21/100...


Analyzing video 31/100...


...

Analyzing video 61/100...


Analyzing video 71/100...


Analyzing video 81/100...


Analyzing video 91/100...



Successfully analyzed 100 videos
Average duration: 5.48 seconds
Min duration: 2.04 seconds
Max duration: 17.68 seconds

Unique resolutions:
  224x224: 98 videos
  576x464: 1 videos
  514x482: 1 videos

Average resolution: 230x229

Average FPS: 25.04
FPS range: 23.98 - 29.97

Video formats:
  .mp4: 21544 videos


## 7. Audio Analysis

In [9]:
def analyze_audio_file(audio_path):
    """
    Analyze a single audio file
    """
    try:
        y, sr = librosa.load(audio_path, sr=None)
        duration = librosa.get_duration(y=y, sr=sr)
        
        return {
            'duration': duration,
            'sample_rate': sr,
            'channels': 1  # librosa loads as mono by default
        }
    except Exception as e:
        return None

def get_all_audio_files(path):
    """
    Get all audio files recursively
    """
    audio_files = []
    audio_extensions = ['.wav', '.mp3', '.flac', '.aac', '.ogg', '.m4a']
    
    for root, dirs, files in os.walk(path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in audio_extensions):
                audio_files.append(os.path.join(root, file))
    
    return audio_files

print("=== AUDIO ANALYSIS ===")
audio_files = get_all_audio_files(DATASET_PATH)
print(f"Total audio files found: {len(audio_files):,}")

# Analyze a sample of audio files (first 100 to avoid timeout)
sample_size = min(100, len(audio_files))
sample_audio = random.sample(audio_files, sample_size) if len(audio_files) > 0 else []

audio_stats = []
audio_durations = []
sample_rates = []
channels = []

for i, audio_path in enumerate(sample_audio):
    if i % 10 == 0:
        print(f"Analyzing audio {i+1}/{len(sample_audio)}...")
    
    stats = analyze_audio_file(audio_path)
    if stats:
        audio_stats.append(stats)
        audio_durations.append(stats['duration'])
        sample_rates.append(stats['sample_rate'])
        channels.append(stats['channels'])

if audio_stats:
    print(f"\nSuccessfully analyzed {len(audio_stats)} audio files")
    print(f"Average duration: {np.mean(audio_durations):.2f} seconds")
    print(f"Min duration: {np.min(audio_durations):.2f} seconds")
    print(f"Max duration: {np.max(audio_durations):.2f} seconds")
    
    # Sample rate analysis
    unique_sr = Counter(sample_rates)
    print(f"\nSample rates:")
    for sr, count in unique_sr.most_common():
        print(f"  {sr} Hz: {count} files")
    
    # Channels analysis
    unique_channels = Counter(channels)
    print(f"\nChannels:")
    for ch, count in unique_channels.most_common():
        print(f"  {ch} (mono): {count} files")
    
    # Audio formats
    formats = Counter([os.path.splitext(a)[1].lower() for a in audio_files])
    print(f"\nAudio formats:")
    for fmt, count in formats.most_common():
        print(f"  {fmt}: {count} files")
else:
    print("No audio files could be analyzed")

=== AUDIO ANALYSIS ===


Total audio files found: 0
No audio files could be analyzed


## 8. Modality Analysis

In [10]:
print("=== MODALITY ANALYSIS ===")

# Count files by modality based on folder structure
modality_counts = {
    'Video+Audio (Fake-Fake)': 0,
    'Video+Audio (Fake-Real)': 0,
    'Video+Audio (Real-Fake)': 0,
    'Video+Audio (Real-Real)': 0,
    'Video Only': 0,
    'Audio Only': 0,
    'Images': 0
}

# Analyze each main category
for category in main_categories:
    category_path = os.path.join(DATASET_PATH, category)
    if os.path.exists(category_path):
        file_count = 0
        for root, dirs, files in os.walk(category_path):
            for file in files:
                ext = os.path.splitext(file)[1].lower()
                if ext in video_extensions:
                    file_count += 1
        
        if 'FakeVideo-FakeAudio' in category:
            modality_counts['Video+Audio (Fake-Fake)'] = file_count
        elif 'FakeVideo-RealAudio' in category:
            modality_counts['Video+Audio (Fake-Real)'] = file_count
        elif 'RealVideo-FakeAudio' in category:
            modality_counts['Video+Audio (Real-Fake)'] = file_count
        elif 'RealVideo-RealAudio' in category:
            modality_counts['Video+Audio (Real-Real)'] = file_count

modality_counts['Video Only'] = video_count
modality_counts['Audio Only'] = audio_count
modality_counts['Images'] = image_count

print("Modality Distribution:")
for modality, count in modality_counts.items():
    print(f"  {modality}: {count:,} files")

print(f"\nTotal multimodal samples (video+audio): {sum([v for k, v in modality_counts.items() if 'Video+Audio' in k]):,}")

=== MODALITY ANALYSIS ===


Modality Distribution:
  Video+Audio (Fake-Fake): 10,835 files
  Video+Audio (Fake-Real): 9,709 files
  Video+Audio (Real-Fake): 500 files
  Video+Audio (Real-Real): 500 files
  Video Only: 21,544 files
  Audio Only: 0 files
  Images: 0 files

Total multimodal samples (video+audio): 21,544


## 9. Metadata Analysis

In [11]:
print("=== DETAILED METADATA ANALYSIS ===")

# Find all metadata files
metadata_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if any(keyword in file.lower() for keyword in ['meta', 'label', 'annotation', 'info']):
            metadata_files.append(os.path.join(root, file))

print(f"Found {len(metadata_files)} potential metadata files:")
for meta_file in metadata_files:
    print(f"  - {meta_file}")

# Analyze the main metadata CSV
if os.path.exists(metadata_path):
    print(f"\n=== DETAILED ANALYSIS OF {os.path.basename(metadata_path)} ===")
    df_meta = pd.read_csv(metadata_path)
    
    print(f"\nFile path: {metadata_path}")
    print(f"Shape: {df_meta.shape[0]} rows x {df_meta.shape[1]} columns")
    
    print(f"\nColumn names and descriptions:")
    for col in df_meta.columns:
        print(f"  - {col}")
        print(f"    Type: {df_meta[col].dtype}")
        print(f"    Unique values: {df_meta[col].nunique()}")
        if df_meta[col].nunique() <= 10:
            print(f"    Values: {df_meta[col].unique()}")
        print(f"    Null count: {df_meta[col].isnull().sum()}")
        print()
    
    print(f"\nFirst 10 rows:")
    print(df_meta.head(10))
    
    print(f"\nLast 10 rows:")
    print(df_meta.tail(10))

=== DETAILED METADATA ANALYSIS ===


Found 1 potential metadata files:
  - d:\uday\Vscode\Projects\Deepfake_Detection\Dataset_FakeAVCeleb\meta_data.csv

=== DETAILED ANALYSIS OF meta_data.csv ===

File path: d:\uday\Vscode\Projects\Deepfake_Detection\Dataset_FakeAVCeleb\meta_data.csv
Shape: 21566 rows x 10 columns

Column names and descriptions:
  - source
    Type: object
    Unique values: 500
    Null count: 0

  - target1
    Type: object
    Unique values: 526
    Null count: 0

  - target2
    Type: object
    Unique values: 513
    Null count: 0

  - method
    Type: object
    Unique values: 7
    Values: ['real' 'rtvc' 'faceswap' 'faceswap-wav2lip' 'fsgan' 'fsgan-wav2lip'
 'wav2lip']
    Null count: 0

  - category
    Type: object
    Unique values: 4
    Values: ['A' 'B' 'C' 'D']
    Null count: 0

  - type
    Type: object
    Unique values: 4
    Values: ['RealVideo-RealAudio' 'RealVideo-FakeAudio' 'FakeVideo-RealAudio'
 'FakeVideo-FakeAudio']
    Null count: 0

  - race
    Type: object
    Unique values: 5


## 10. Deepfake Categories Analysis

In [12]:
print("=== DEEPFAKE CATEGORIES ANALYSIS ===")

# Based on folder structure, identify manipulation types
print("Manipulation types based on folder structure:")

manipulation_types = {
    'FakeVideo-FakeAudio': 'Both video and audio are fake',
    'FakeVideo-RealAudio': 'Video is fake, audio is real',
    'RealVideo-FakeAudio': 'Video is real, audio is fake',
    'RealVideo-RealAudio': 'Both video and audio are real (ground truth)'
}

for category, description in manipulation_types.items():
    count = class_stats.get(category, 0)
    print(f"  {category}: {count:,} samples")
    print(f"    Description: {description}")

print("\n=== SUB-CATEGORY ANALYSIS ===")
# Check for ethnic/gender subcategories
for category in main_categories:
    category_path = os.path.join(DATASET_PATH, category)
    if os.path.exists(category_path):
        print(f"\n{category} subcategories:")
        subdirs = [d for d in os.listdir(category_path) if os.path.isdir(os.path.join(category_path, d))]
        for subdir in sorted(subdirs):
            subdir_path = os.path.join(category_path, subdir)
            file_count = sum(len(files) for r, d, files in os.walk(subdir_path))
            print(f"  - {subdir}/: {file_count:,} files")

=== DEEPFAKE CATEGORIES ANALYSIS ===
Manipulation types based on folder structure:
  FakeVideo-FakeAudio: 10,835 samples
    Description: Both video and audio are fake
  FakeVideo-RealAudio: 9,709 samples
    Description: Video is fake, audio is real
  RealVideo-FakeAudio: 500 samples
    Description: Video is real, audio is fake
  RealVideo-RealAudio: 1,000 samples
    Description: Both video and audio are real (ground truth)

=== SUB-CATEGORY ANALYSIS ===

FakeVideo-FakeAudio subcategories:
  - African/: 2,035 files
  - Asian (East)/: 1,649 files
  - Asian (South)/: 2,262 files
  - Caucasian (American)/: 2,505 files
  - Caucasian (European)/: 2,384 files

FakeVideo-RealAudio subcategories:
  - African/: 1,854 files
  - Asian (East)/: 1,635 files
  - Asian (South)/: 1,952 files
  - Caucasian (American)/: 2,159 files
  - Caucasian (European)/: 2,109 files

RealVideo-FakeAudio subcategories:
  - African/: 100 files
  - Asian (East)/: 100 files
  - Asian (South)/: 100 files
  - Caucasian

  - African/: 200 files
  - Asian (East)/: 200 files
  - Asian (South)/: 200 files
  - Caucasian (American)/: 200 files
  - Caucasian (European)/: 200 files


## 11. Train/Test Split Analysis

In [13]:
print("=== TRAIN/TEST SPLIT ANALYSIS ===")

# Check for train/test folders
train_test_keywords = ['train', 'test', 'val', 'validation', 'split']
train_test_folders = []

for root, dirs, files in os.walk(DATASET_PATH):
    for dir_name in dirs:
        if any(keyword in dir_name.lower() for keyword in train_test_keywords):
            train_test_folders.append(os.path.join(root, dir_name))

if train_test_folders:
    print(f"Found {len(train_test_folders)} train/test related folders:")
    for folder in train_test_folders:
        file_count = sum(len(files) for r, d, files in os.walk(folder))
        print(f"  - {folder}: {file_count:,} files")
else:
    print("No official train/test split folders found")

# Check metadata for split information
if os.path.exists(metadata_path):
    df_meta = pd.read_csv(metadata_path)
    split_columns = [col for col in df_meta.columns if any(keyword in col.lower() for keyword in ['split', 'train', 'test', 'val'])]
    
    if split_columns:
        print(f"\nMetadata contains split information in columns: {split_columns}")
        for col in split_columns:
            print(f"\n  {col} distribution:")
            print(df_meta[col].value_counts())
    else:
        print("\nMetadata does not contain explicit train/test split columns")

=== TRAIN/TEST SPLIT ANALYSIS ===


No official train/test split folders found

Metadata does not contain explicit train/test split columns


## 12. Sample Inspection

In [14]:
print("=== SAMPLE INSPECTION ===")

# Get real samples (from RealVideo-RealAudio)
real_samples_path = os.path.join(DATASET_PATH, 'RealVideo-RealAudio')
fake_samples_paths = [
    os.path.join(DATASET_PATH, 'FakeVideo-FakeAudio'),
    os.path.join(DATASET_PATH, 'FakeVideo-RealAudio'),
    os.path.join(DATASET_PATH, 'RealVideo-FakeAudio')
]

# Collect real samples
real_video_files = []
if os.path.exists(real_samples_path):
    for root, dirs, files in os.walk(real_samples_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in video_extensions):
                real_video_files.append(os.path.join(root, file))

# Collect fake samples
fake_video_files = []
for fake_path in fake_samples_paths:
    if os.path.exists(fake_path):
        for root, dirs, files in os.walk(fake_path):
            for file in files:
                if any(file.lower().endswith(ext) for ext in video_extensions):
                    fake_video_files.append(os.path.join(root, file))

print(f"Total real samples available: {len(real_video_files):,}")
print(f"Total fake samples available: {len(fake_video_files):,}")

# Select random samples
num_samples = min(10, len(real_video_files))
sample_real = random.sample(real_video_files, num_samples) if real_video_files else []

num_fake_samples = min(10, len(fake_video_files))
sample_fake = random.sample(fake_video_files, num_fake_samples) if fake_video_files else []

print(f"\n=== 10 RANDOM REAL SAMPLES ===")
for i, sample_path in enumerate(sample_real[:10], 1):
    rel_path = os.path.relpath(sample_path, DATASET_PATH)
    stats = analyze_video_file(sample_path)
    if stats:
        print(f"{i}. {rel_path}")
        print(f"   Label: Real")
        print(f"   Duration: {stats['duration']:.2f}s")
        print(f"   Resolution: {stats['resolution'][0]}x{stats['resolution'][1]}")
        print(f"   FPS: {stats['fps']:.2f}")
    else:
        print(f"{i}. {rel_path}")
        print(f"   Label: Real")
        print(f"   Status: Could not analyze")

print(f"\n=== 10 RANDOM FAKE SAMPLES ===")
for i, sample_path in enumerate(sample_fake[:10], 1):
    rel_path = os.path.relpath(sample_path, DATASET_PATH)
    stats = analyze_video_file(sample_path)
    if stats:
        # Determine fake type from path
        fake_type = "Unknown"
        if 'FakeVideo-FakeAudio' in sample_path:
            fake_type = "Fake Video + Fake Audio"
        elif 'FakeVideo-RealAudio' in sample_path:
            fake_type = "Fake Video + Real Audio"
        elif 'RealVideo-FakeAudio' in sample_path:
            fake_type = "Real Video + Fake Audio"
        
        print(f"{i}. {rel_path}")
        print(f"   Label: Fake ({fake_type})")
        print(f"   Duration: {stats['duration']:.2f}s")
        print(f"   Resolution: {stats['resolution'][0]}x{stats['resolution'][1]}")
        print(f"   FPS: {stats['fps']:.2f}")
    else:
        print(f"{i}. {rel_path}")
        print(f"   Label: Fake")
        print(f"   Status: Could not analyze")

=== SAMPLE INSPECTION ===


Total real samples available: 500
Total fake samples available: 21,044

=== 10 RANDOM REAL SAMPLES ===
1. RealVideo-RealAudio\African\men\id00830\00143.mp4
   Label: Real
   Duration: 12.48s
   Resolution: 224x224
   FPS: 25.00
2. RealVideo-RealAudio\Asian (East)\men\id08652\00006.mp4
   Label: Real
   Duration: 4.16s
   Resolution: 224x224
   FPS: 25.00
3. RealVideo-RealAudio\Asian (East)\women\id06462\00014.mp4
   Label: Real
   Duration: 4.72s
   Resolution: 224x224
   FPS: 25.00
4. RealVideo-RealAudio\African\women\id00829\00271.mp4
   Label: Real
   Duration: 7.80s
   Resolution: 224x224
   FPS: 25.00
5. RealVideo-RealAudio\Asian (East)\men\id06535\00183.mp4
   Label: Real
   Duration: 6.20s
   Resolution: 224x224
   FPS: 25.00
6. RealVideo-RealAudio\Asian (South)\men\id00350\00015.mp4
   Label: Real
   Duration: 3.96s
   Resolution: 224x224
   FPS: 25.00


7. RealVideo-RealAudio\Asian (East)\women\id09125\00098.mp4
   Label: Real
   Duration: 6.84s
   Resolution: 224x224
   FPS: 25.00
8. RealVideo-RealAudio\Caucasian (European)\men\id00368\00078.mp4
   Label: Real
   Duration: 7.64s
   Resolution: 224x224
   FPS: 25.00
9. RealVideo-RealAudio\Asian (East)\men\id04774\00032.mp4
   Label: Real
   Duration: 4.48s
   Resolution: 224x224
   FPS: 25.00
10. RealVideo-RealAudio\Caucasian (European)\men\id00282\00268.mp4
   Label: Real
   Duration: 6.60s
   Resolution: 224x224
   FPS: 25.00

=== 10 RANDOM FAKE SAMPLES ===
1. FakeVideo-FakeAudio\African\men\id00987\00160_id04727_wavtolip.mp4
   Label: Fake (Fake Video + Fake Audio)
   Duration: 3.72s
   Resolution: 224x224
   FPS: 25.00
2. FakeVideo-FakeAudio\Asian (South)\women\id07686\00254_id06437_3BZvvzBhj8s_id00043_wavtolip.mp4
   Label: Fake (Fake Video + Fake Audio)
   Duration: 5.16s
   Resolution: 224x224
   FPS: 25.00
3. FakeVideo-RealAudio\African\women\id04540\00078_id05235_wavtolip.mp4

5. FakeVideo-FakeAudio\Caucasian (American)\men\id00696\00005_id04034_wavtolip.mp4
   Label: Fake (Fake Video + Fake Audio)
   Duration: 4.92s
   Resolution: 224x224
   FPS: 25.00
6. FakeVideo-FakeAudio\Caucasian (European)\women\id00826\00065_id00735_ivJSkq5hGOQ_id03589_wavtolip.mp4
   Label: Fake (Fake Video + Fake Audio)
   Duration: 5.64s
   Resolution: 224x224
   FPS: 25.00
7. FakeVideo-FakeAudio\African\women\id01838\00126_id05106_ZnnDdjlABM4_id01907_wavtolip.mp4
   Label: Fake (Fake Video + Fake Audio)
   Duration: 4.84s
   Resolution: 224x224
   FPS: 25.00
8. FakeVideo-RealAudio\Caucasian (European)\women\id00491\00122_id01002_eQTviHjyISo.mp4
   Label: Fake (Fake Video + Real Audio)
   Duration: 4.60s
   Resolution: 224x224
   FPS: 25.00
9. FakeVideo-FakeAudio\African\men\id02296\00019_id00366_wavtolip.mp4
   Label: Fake (Fake Video + Fake Audio)
   Duration: 5.00s
   Resolution: 224x224
   FPS: 25.00
10. FakeVideo-FakeAudio\African\men\id00478\00206_id01544_wavtolip.mp4
   Lab

## 13. Potential Issues Analysis

In [15]:
print("=== POTENTIAL ISSUES ANALYSIS ===")

# 1. Check for empty folders
print("1. Checking for empty folders...")
empty_folders = []
for root, dirs, files in os.walk(DATASET_PATH):
    for dir_name in dirs:
        dir_path = os.path.join(root, dir_name)
        if not os.listdir(dir_path):
            empty_folders.append(dir_path)

if empty_folders:
    print(f"   Found {len(empty_folders)} empty folders:")
    for folder in empty_folders[:10]:  # Show first 10
        print(f"     - {folder}")
else:
    print("   No empty folders found")

# 2. Check for corrupted files (sample check)
print("\n2. Checking for corrupted files (sample of 50 files)...")
corrupted_files = []
sample_files = random.sample(video_files + audio_files, min(50, len(video_files + audio_files)))

for file_path in sample_files:
    try:
        if file_path.endswith(tuple(video_extensions)):
            cap = cv2.VideoCapture(file_path)
            if not cap.isOpened():
                corrupted_files.append(file_path)
            cap.release()
        elif file_path.endswith(tuple(audio_extensions)):
            y, sr = librosa.load(file_path, sr=None)
    except:
        corrupted_files.append(file_path)

if corrupted_files:
    print(f"   Found {len(corrupted_files)} potentially corrupted files:")
    for file in corrupted_files[:10]:
        print(f"     - {file}")
else:
    print("   No corrupted files detected in sample")

# 3. Check for duplicate files (by name)
print("\n3. Checking for duplicate file names...")
all_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        all_files.append(file)

file_counts = Counter(all_files)
duplicates = {name: count for name, count in file_counts.items() if count > 1}

if duplicates:
    print(f"   Found {len(duplicates)} duplicate file names:")
    for name, count in list(duplicates.items())[:10]:
        print(f"     - {name}: {count} occurrences")
else:
    print("   No duplicate file names found")

# 4. Class imbalance check
print("\n4. Checking for class imbalance...")
if total_real > 0 and total_fake > 0:
    ratio = total_real / total_fake
    print(f"   Real-to-Fake ratio: {ratio:.2f}")
    if ratio < 0.5 or ratio > 2.0:
        print("   WARNING: Significant class imbalance detected!")
    else:
        print("   Class distribution is reasonably balanced")
else:
    print("   Unable to calculate class imbalance")

# 5. Missing files check (if metadata exists)
print("\n5. Checking for missing files referenced in metadata...")
if os.path.exists(metadata_path):
    df_meta = pd.read_csv(metadata_path)
    # Check if there's a file path column
    path_columns = [col for col in df_meta.columns if any(keyword in col.lower() for keyword in ['path', 'file', 'name'])]
    if path_columns:
        print(f"   Found path columns: {path_columns}")
        # This would require custom logic based on actual column names
        print("   (Detailed missing file check requires manual inspection of path columns)")
    else:
        print("   No file path columns found in metadata")
else:
    print("   No metadata file available for missing file check")

=== POTENTIAL ISSUES ANALYSIS ===
1. Checking for empty folders...


   No empty folders found

2. Checking for corrupted files (sample of 50 files)...


   No corrupted files detected in sample

3. Checking for duplicate file names...


   Found 2243 duplicate file names:
     - 00109_id00391_wavtolip.mp4: 2 occurrences
     - 00109_id00475_wavtolip.mp4: 2 occurrences
     - 00109_id00701_wavtolip.mp4: 2 occurrences
     - 00109_id01637_wavtolip.mp4: 2 occurrences
     - 00109_id02005_wavtolip.mp4: 3 occurrences
     - 00109_id02316_wavtolip.mp4: 4 occurrences
     - 00010_id00366_wavtolip.mp4: 4 occurrences
     - 00010_id01076_wavtolip.mp4: 3 occurrences
     - 00010_id01179_wavtolip.mp4: 2 occurrences
     - 00010_id01392_wavtolip.mp4: 4 occurrences

4. Checking for class imbalance...
   Real-to-Fake ratio: 0.07

5. Checking for missing files referenced in metadata...
   Found path columns: ['path', 'Unnamed: 9']
   (Detailed missing file check requires manual inspection of path columns)


## 14. Final Summary Report

In [16]:
print("=" * 80)
print("FAKEAVCELEB DATASET AUDIT REPORT")
print("=" * 80)
print(f"Date: June 22, 2026")
print(f"Dataset Path: {DATASET_PATH}")
print("=" * 80)

print("\nA. EXACT DATASET STRUCTURE")
print("-" * 80)
print("Top-level folders:")
for folder in os.listdir(DATASET_PATH):
    if os.path.isdir(os.path.join(DATASET_PATH, folder)):
        print(f"  📁 {folder}/")

print("\nMain categories:")
for category in main_categories:
    count = class_stats.get(category, 0)
    print(f"  • {category}: {count:,} files")

print("\nB. DATASET STATISTICS")
print("-" * 80)
print(f"Total files: {total_files:,}")
print(f"Video files: {video_count:,}")
print(f"Audio files: {audio_count:,}")
print(f"Image files: {image_count:,}")
print(f"Total dataset size: {total_size / (1024**3):.2f} GB")
print(f"Average file size: {total_size / total_files / (1024**2):.2f} MB")

print("\nC. LABEL INFORMATION")
print("-" * 80)
print("Labels are stored via:")
print("  • Folder structure (4 main categories)")
print("  • Metadata CSV file (meta_data.csv)")
print(f"  • Total CSV files: {len(csv_files)}")
print(f"  • Total JSON files: {len(json_files)}")

print("\nD. CLASS DISTRIBUTION")
print("-" * 80)
print(f"Real samples: {total_real:,}")
print(f"Fake samples: {total_fake:,}")
print(f"Real-to-Fake ratio: {total_real}:{total_fake} ({total_real/total_fake:.2f})")
print(f"Total samples: {total_samples:,}")

print("\nE. VIDEO ANALYSIS")
print("-" * 80)
if video_stats:
    print(f"Total videos analyzed: {len(video_stats)}")
    print(f"Average duration: {np.mean(video_durations):.2f} seconds")
    print(f"Duration range: {np.min(video_durations):.2f} - {np.max(video_durations):.2f} seconds")
    print(f"Average resolution: {np.mean([r[0] for r in resolutions]):.0f}x{np.mean([r[1] for r in resolutions]):.0f}")
    print(f"Unique resolutions: {len(unique_resolutions)}")
    print(f"Average FPS: {np.mean(valid_fps):.2f}" if valid_fps else "FPS data unavailable")
    print(f"Video formats: {list(formats.keys())}")
else:
    print("Video analysis data unavailable")

print("\nF. AUDIO ANALYSIS")
print("-" * 80)
if audio_stats:
    print(f"Total audio files analyzed: {len(audio_stats)}")
    print(f"Average duration: {np.mean(audio_durations):.2f} seconds")
    print(f"Duration range: {np.min(audio_durations):.2f} - {np.max(audio_durations):.2f} seconds")
    print(f"Sample rates: {list(unique_sr.keys())}")
    print(f"Channels: Mono (all files)")
    print(f"Audio formats: {list(formats.keys())}")
else:
    print("Audio analysis data unavailable")

print("\nG. MODALITY ANALYSIS")
print("-" * 80)
for modality, count in modality_counts.items():
    print(f"  • {modality}: {count:,}")
print(f"\nDataset contains: Video + Audio multimodal data")

print("\nH. DEEPFAKE CATEGORIES")
print("-" * 80)
print("Manipulation types available:")
for category, description in manipulation_types.items():
    count = class_stats.get(category, 0)
    print(f"  • {category}: {count:,} samples")
    print(f"    {description}")

print("\nI. TRAIN/TEST SPLIT")
print("-" * 80)
if train_test_folders:
    print(f"Official splits found: {len(train_test_folders)} folders")
    for folder in train_test_folders:
        print(f"  • {folder}")
else:
    print("No official train/test split folders detected")
    print("Recommendation: Create custom splits based on requirements")

print("\nJ. POTENTIAL ISSUES")
print("-" * 80)
print(f"Empty folders: {len(empty_folders)}")
print(f"Corrupted files (sample): {len(corrupted_files)}")
print(f"Duplicate file names: {len(duplicates)}")
if total_real > 0 and total_fake > 0:
    ratio = total_real / total_fake
    if ratio < 0.5 or ratio > 2.0:
        print(f"Class imbalance: YES (ratio: {ratio:.2f})")
    else:
        print(f"Class imbalance: NO (ratio: {ratio:.2f})")

print("\n" + "=" * 80)
print("RECOMMENDATIONS")
print("=" * 80)

print("\nA. TRAINING STRATEGY")
print("-" * 80)
print("1. For Video-only models:")
print("   • Use all 4 categories")
print("   • Focus on video content analysis")
print("   • Consider frame-level or clip-level processing")
print("")
print("2. For Audio-only models:")
print("   • Use FakeVideo-FakeAudio, FakeVideo-RealAudio, RealVideo-FakeAudio")
print("   • Extract audio tracks from videos")
print("   • Focus on spectral analysis")
print("")
print("3. For Fusion models (Video + Audio):")
print("   • Ideal for this dataset")
print("   • Leverage multimodal nature")
print("   • Combine video and audio features")
print("   • Use RealVideo-RealAudio as ground truth")

print("\nB. DATASET SUFFICIENCY")
print("-" * 80)
print("Video model: " + ("SUFFICIENT" if video_count > 1000 else "MAY NEED MORE DATA"))
print("Audio model: " + ("SUFFICIENT" if audio_count > 1000 else "MAY NEED MORE DATA"))
print("Fusion model: " + ("SUFFICIENT" if total_samples > 1000 else "MAY NEED MORE DATA"))

print("\nC. CONCERNS BEFORE TRAINING")
print("-" * 80)
print("1. No official train/test split - need to create custom splits")
print("2. Verify all files are readable before training")
print("3. Check for class imbalance and apply balancing if needed")
print("4. Ensure consistent video/audio quality across categories")
print("5. Validate metadata completeness and accuracy")

print("\n" + "=" * 80)
print("END OF REPORT")


FAKEAVCELEB DATASET AUDIT REPORT
Date: June 22, 2026
Dataset Path: d:\uday\Vscode\Projects\Deepfake_Detection\Dataset_FakeAVCeleb

A. EXACT DATASET STRUCTURE
--------------------------------------------------------------------------------
Top-level folders:
  📁 FakeVideo-FakeAudio/
  📁 FakeVideo-RealAudio/
  📁 RealVideo-FakeAudio/
  📁 RealVideo-RealAudio/

Main categories:
  • FakeVideo-FakeAudio: 10,835 files
  • FakeVideo-RealAudio: 9,709 files
  • RealVideo-FakeAudio: 500 files
  • RealVideo-RealAudio: 1,000 files

B. DATASET STATISTICS
--------------------------------------------------------------------------------
Total files: 22,046
Video files: 21,544
Audio files: 0
Image files: 0
Total dataset size: 6.16 GB
Average file size: 0.29 MB

C. LABEL INFORMATION
--------------------------------------------------------------------------------
Labels are stored via:
  • Folder structure (4 main categories)
  • Metadata CSV file (meta_data.csv)
  • Total CSV files: 1
  • Total JSON files

## 15. Audio Extraction Audit

In [17]:
from moviepy import VideoFileClip
import random
from collections import Counter
import numpy as np
import os

print("=== AUDIO EXTRACTION AUDIT ===")
sample_size = 100
random.seed(42)

# all_videos should be available from previous cells. 
# If not, let's redefine it just in case.
def get_all_videos(path):
    video_files = []
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv']
    for root, dirs, files in os.walk(path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in video_extensions):
                video_files.append(os.path.join(root, file))
    return video_files

all_videos = get_all_videos(DATASET_PATH)
audit_videos = random.sample(all_videos, min(sample_size, len(all_videos))) if all_videos else []

success_count = 0
missing_audio_count = 0
extraction_errors = 0

sample_rates = []
durations = []
channels = []
examples = []

print(f"Testing {len(audit_videos)} random videos for audio extraction...\n")

for i, video_path in enumerate(audit_videos):
    rel_path = os.path.relpath(video_path, DATASET_PATH)
    try:
        video = VideoFileClip(video_path)
        if video.audio is None:
            missing_audio_count += 1
            if len(examples) < 20:
                examples.append({'path': rel_path, 'status': 'Missing Audio'})
        else:
            # Extract audio info
            sr = video.audio.fps
            ch = video.audio.nchannels
            duration = video.audio.duration
            
            success_count += 1
            sample_rates.append(sr)
            channels.append(ch)
            durations.append(duration)
            
            if len(examples) < 20:
                examples.append({
                    'path': rel_path, 
                    'status': 'Success', 
                    'sr': sr, 
                    'channels': ch, 
                    'duration': duration
                })
        video.close()
    except Exception as e:
        extraction_errors += 1
        if len(examples) < 20:
            examples.append({'path': rel_path, 'status': f'Extraction Error: {str(e)}'})

print("=== 20 RANDOM EXAMPLES ===")
for i, ex in enumerate(examples[:20], 1):
    if ex['status'] == 'Success':
        ch_str = 'Stereo' if ex['channels'] == 2 else ('Mono' if ex['channels'] == 1 else str(ex['channels']))
        print(f"{i}. {ex['path']} -> {ex['status']} | {ex['sr']}Hz | {ch_str} | {ex['duration']:.2f}s")
    else:
        print(f"{i}. {ex['path']} -> {ex['status']}")

print("\n=== OVERALL AUDIO EXTRACTION STATISTICS ===")
print(f"Total Videos Checked: {len(audit_videos)}")
print(f"Successfully Extracted: {success_count}")
print(f"Missing Audio: {missing_audio_count}")
print(f"Extraction Errors: {extraction_errors}")

if success_count > 0:
    print(f"\nAverage Audio Duration: {np.mean(durations):.2f} seconds")
    print(f"Min Duration: {np.min(durations):.2f} seconds")
    print(f"Max Duration: {np.max(durations):.2f} seconds")
    
    print("\nSample Rate Distribution:")
    for sr, count in Counter(sample_rates).most_common():
        print(f"  {sr} Hz: {count} videos")
        
    print("\nChannel Distribution:")
    for ch, count in Counter(channels).most_common():
        ch_str = 'Stereo' if ch == 2 else ('Mono' if ch == 1 else str(ch))
        print(f"  {ch_str}: {count} videos")


=== AUDIO EXTRACTION AUDIT ===


Testing 100 random videos for audio extraction...



=== 20 RANDOM EXAMPLES ===
1. RealVideo-FakeAudio\Caucasian (European)\men\id00225\00078_fake.mp4 -> Success | 44100Hz | Stereo | 13.52s
2. FakeVideo-FakeAudio\Asian (East)\women\id09174\00015_id02587_KzwJPH5_8j0_faceswap_id06060_wavtolip.mp4 -> Success | 44100Hz | Stereo | 4.14s
3. FakeVideo-FakeAudio\African\men\id01779\00010_id01691_IVtS5z8Jrrk_faceswap_id01779_wavtolip.mp4 -> Success | 44100Hz | Stereo | 5.25s
4. FakeVideo-FakeAudio\Caucasian (European)\men\id00519\00028_id01098_wavtolip.mp4 -> Success | 44100Hz | Stereo | 3.39s
5. FakeVideo-FakeAudio\Caucasian (American)\women\id01005\00028_id01225_wavtolip.mp4 -> Success | 44100Hz | Stereo | 5.60s
6. FakeVideo-FakeAudio\Caucasian (American)\women\id00097\00162_id00180_lkeAa8Od_tg_id00231_wavtolip.mp4 -> Success | 44100Hz | Stereo | 4.42s
7. FakeVideo-FakeAudio\Asian (South)\men\id07179\00206_id07165_oJ_iy16IBTM_faceswap_id07233_wavtolip.mp4 -> Success | 44100Hz | Stereo | 4.04s
8. FakeVideo-FakeAudio\Asian (East)\women\id06066\00

## 16. Identity Leakage Audit

In [18]:
import pandas as pd
import numpy as np
import os

print("=== IDENTITY LEAKAGE AUDIT ===")

meta_path = os.path.join(DATASET_PATH, 'meta_data.csv')
if os.path.exists(meta_path):
    df = pd.read_csv(meta_path)
    
    # 1. Total unique identities
    sources = set(df['source'].dropna().unique())
    targets = set(df['target1'].dropna().unique()).union(set(df['target2'].dropna().unique()))
    targets.discard('-')
    
    print(f"Total unique source identities: {len(sources)}")
    print(f"Total unique target identities: {len(targets)}")
    
    # 2. Number of identities in each category
    print("\nNumber of source identities appearing in each category (type):")
    for t in df['type'].unique():
        count = df[df['type'] == t]['source'].nunique()
        print(f"  - {t}: {count} identities")
        
    # 3. Same identity in Real vs Fake
    real_sources = set(df[df['type'] == 'RealVideo-RealAudio']['source'].unique())
    fake_sources = set(df[df['type'] != 'RealVideo-RealAudio']['source'].unique())
    overlap = real_sources.intersection(fake_sources)
    
    print(f"\nDoes the same identity appear in BOTH real and fake samples? {'YES' if len(overlap) > 0 else 'NO'}")
    if overlap:
        print(f"  - Found {len(overlap)} identities appearing in both real and fake sets.")
        print(f"  - Examples: {list(overlap)[:5]}")
        
    # 4. Identity frequency
    counts = df['source'].value_counts()
    print("\nDoes the same source identity appear hundreds of times? ")
    max_count = counts.max()
    print(f"  - YES, the most frequent identity appears {max_count} times." if max_count >= 100 else f"  - NO, max occurrences is {max_count}.")
    
    print("\nTop 20 most frequent source identities:")
    for identity, count in counts.head(20).items():
        print(f"  - {identity}: {count} times")
        
    # 5. Recommended splitting strategy
    print("\n=== RECOMMENDED SPLITTING STRATEGY ===")
    if len(overlap) > 0 or max_count > 1:
        print("RECOMMENDATION: **IDENTITY-BASED SPLIT**")
        print("Reasoning: ")
        if len(overlap) > 0:
            print("  - Identities appear in both real and fake samples.")
        if max_count > 1:
            print("  - Identities appear multiple times across samples.")
        print("  - A random split will cause identity leakage, allowing the model to cheat by memorizing faces/voices.")
        
        # Suggest identities
        np.random.seed(42)
        all_sources = sorted(list(sources))
        np.random.shuffle(all_sources)
        
        n_train = int(len(all_sources) * 0.7)
        n_val = int(len(all_sources) * 0.15)
        
        train_ids = all_sources[:n_train]
        val_ids = all_sources[n_train:n_train+n_val]
        test_ids = all_sources[n_train+n_val:]
        
        print(f"\nSuggested Identity Split (70/15/15):")
        print(f"  - Train identities: {len(train_ids)} ({len(train_ids)/len(all_sources)*100:.1f}%)")
        print(f"    Examples: {train_ids[:5]}...")
        print(f"  - Validation identities: {len(val_ids)} ({len(val_ids)/len(all_sources)*100:.1f}%)")
        print(f"    Examples: {val_ids[:5]}...")
        print(f"  - Test identities: {len(test_ids)} ({len(test_ids)/len(all_sources)*100:.1f}%)")
        print(f"    Examples: {test_ids[:5]}...")
    else:
        print("RECOMMENDATION: Random split is acceptable as no significant leakage risk was detected.")
else:
    print("Error: meta_data.csv not found.")


=== IDENTITY LEAKAGE AUDIT ===
Total unique source identities: 500
Total unique target identities: 578

Number of source identities appearing in each category (type):
  - RealVideo-RealAudio: 500 identities
  - RealVideo-FakeAudio: 500 identities
  - FakeVideo-RealAudio: 500 identities
  - FakeVideo-FakeAudio: 499 identities

Does the same identity appear in BOTH real and fake samples? YES
  - Found 500 identities appearing in both real and fake sets.
  - Examples: ['id06152', 'id03344', 'id01933', 'id00078', 'id00071']



Does the same source identity appear hundreds of times? 
  - NO, max occurrences is 88.

Top 20 most frequent source identities:
  - id00076: 88 times
  - id01182: 68 times
  - id00520: 67 times
  - id03525: 66 times
  - id00183: 66 times
  - id00264: 66 times
  - id01210: 66 times
  - id00841: 65 times
  - id04073: 65 times
  - id03757: 65 times
  - id00100: 64 times
  - id01048: 64 times
  - id01058: 63 times
  - id00068: 63 times
  - id01075: 63 times
  - id07136: 61 times
  - id04526: 61 times
  - id04537: 60 times
  - id07689: 60 times
  - id00032: 60 times

=== RECOMMENDED SPLITTING STRATEGY ===
RECOMMENDATION: **IDENTITY-BASED SPLIT**
Reasoning: 
  - Identities appear in both real and fake samples.
  - Identities appear multiple times across samples.
  - A random split will cause identity leakage, allowing the model to cheat by memorizing faces/voices.

Suggested Identity Split (70/15/15):
  - Train identities: 350 (70.0%)
    Examples: ['id04374', 'id00345', 'id04561', 'id0075

## 17. Label Integrity Audit

## 17. Label Integrity Audit

In [21]:
import pandas as pd
import os
import random

print("=== LABEL INTEGRITY AUDIT ===")
meta_path = os.path.join(DATASET_PATH, 'meta_data.csv')
if os.path.exists(meta_path):
    df = pd.read_csv(meta_path)
    
    # Get all videos
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv']
    videos = []
    for root, dirs, files in os.walk(DATASET_PATH):
        for file in files:
            if any(file.lower().endswith(ext) for ext in video_extensions):
                videos.append(os.path.join(root, file))
                
    sample_size = min(500, len(videos))
    random.seed(123)
    audit_videos = random.sample(videos, sample_size)
    
    mismatches = []
    print(f"Checking {sample_size} random samples for label integrity...\n")
    
    for video_path in audit_videos:
        rel_path = os.path.relpath(video_path, DATASET_PATH)
        parts = rel_path.split(os.sep)
        folder_label = parts[0]
        
        filename = os.path.basename(video_path)
        source_id = parts[-2] if len(parts) >= 2 else None
        
        match = df[(df['path'] == filename) & (df['source'] == source_id)]
        if len(match) >= 1:
            meta_type = match.iloc[0]['type']
            if meta_type != folder_label:
                mismatches.append({'file': rel_path, 'folder_label': folder_label, 'meta_type': meta_type})
        else:
            match_fallback = df[df['path'] == filename]
            if len(match_fallback) == 1:
                meta_type = match_fallback.iloc[0]['type']
                if meta_type != folder_label:
                    mismatches.append({'file': rel_path, 'folder_label': folder_label, 'meta_type': meta_type})
                
    print(f"Total Mismatches Found: {len(mismatches)}")
    if mismatches:
        print("\nExamples of mismatches:")
        for m in mismatches[:10]:
            print(f"  - {m['file']} -> Folder: {m['folder_label']}, Meta: {m['meta_type']}")
else:
    print("Required data not found.")


=== LABEL INTEGRITY AUDIT ===


Checking 500 random samples for label integrity...



Total Mismatches Found: 24

Examples of mismatches:
  - FakeVideo-FakeAudio\Asian (South)\women\id05434\00052_id00488_wavtolip.mp4 -> Folder: FakeVideo-FakeAudio, Meta: FakeVideo-RealAudio
  - FakeVideo-FakeAudio\African\men\id00478\00206_id01544_wavtolip.mp4 -> Folder: FakeVideo-FakeAudio, Meta: FakeVideo-RealAudio
  - FakeVideo-FakeAudio\African\men\id01598\00044_id00987_wavtolip.mp4 -> Folder: FakeVideo-FakeAudio, Meta: FakeVideo-RealAudio
  - FakeVideo-FakeAudio\Asian (South)\women\id06428\00043_id05845_wavtolip.mp4 -> Folder: FakeVideo-FakeAudio, Meta: FakeVideo-RealAudio
  - FakeVideo-FakeAudio\African\men\id01972\00078_id00781_wavtolip.mp4 -> Folder: FakeVideo-FakeAudio, Meta: FakeVideo-RealAudio
  - FakeVideo-FakeAudio\Caucasian (American)\women\id00100\00028_id00180_wavtolip.mp4 -> Folder: FakeVideo-FakeAudio, Meta: FakeVideo-RealAudio
  - FakeVideo-FakeAudio\Caucasian (European)\men\id00225\00078_id00358_wavtolip.mp4 -> Folder: FakeVideo-FakeAudio, Meta: FakeVideo-RealAudio
 

## 18. Identity Split Statistics

In [22]:
import pandas as pd
import numpy as np

print("=== IDENTITY SPLIT STATISTICS ===")
if os.path.exists(meta_path):
    df = pd.read_csv(meta_path)
    
    sources = set(df['source'].dropna().unique())
    all_sources = sorted(list(sources))
    np.random.seed(42)
    np.random.shuffle(all_sources)
    
    n_train = 350
    n_val = 75
    train_ids = all_sources[:n_train]
    val_ids = all_sources[n_train:n_train+n_val]
    test_ids = all_sources[n_train+n_val:]
    
    def get_split_stats(split_name, ids_list):
        split_df = df[df['source'].isin(ids_list)]
        n_videos = len(split_df)
        n_real = len(split_df[split_df['type'] == 'RealVideo-RealAudio'])
        n_fake = n_videos - n_real
        
        print(f"\n{split_name} Split ({len(ids_list)} identities):")
        print(f"  - Total Videos: {n_videos}")
        print(f"  - Real Samples: {n_real}")
        print(f"  - Fake Samples: {n_fake}")
        if n_videos > 0:
            print(f"  - Real-to-Fake Ratio: {n_real/n_fake if n_fake > 0 else 'All Real':.4f}")
            
    get_split_stats('Train', train_ids)
    get_split_stats('Validation', val_ids)
    get_split_stats('Test', test_ids)
    
    print("\nCheck completed. Ensuring test set is appropriately balanced.")
else:
    print("Error: meta_data.csv not found.")


=== IDENTITY SPLIT STATISTICS ===

Train Split (350 identities):
  - Total Videos: 15099
  - Real Samples: 350
  - Fake Samples: 14749
  - Real-to-Fake Ratio: 0.0237

Validation Split (75 identities):
  - Total Videos: 3194
  - Real Samples: 75
  - Fake Samples: 3119
  - Real-to-Fake Ratio: 0.0240

Test Split (75 identities):
  - Total Videos: 3273
  - Real Samples: 75
  - Fake Samples: 3198
  - Real-to-Fake Ratio: 0.0235

Check completed. Ensuring test set is appropriately balanced.


## 17. Label Integrity Audit

In [ ]:
import pandas as pd
import os
from collections import Counter

print("=== DUPLICATE FILENAME INVESTIGATION ===")

meta_path = os.path.join(DATASET_PATH, 'meta_data.csv')
df = pd.read_csv(meta_path)

filename_counts = Counter(df['path'])

duplicates = {k: v for k, v in filename_counts.items() if v > 1}

print(f"Total metadata rows: {len(df)}")
print(f"Unique filenames: {len(filename_counts)}")
print(f"Duplicate filenames: {len(duplicates)}")

print("\n=== TOP 20 DUPLICATE FILENAMES ===")

for filename, count in sorted(
        duplicates.items(),
        key=lambda x: x[1],
        reverse=True
    )[:20]:

    print(f"\nFilename: {filename}")
    print(f"Occurrences: {count}")

    rows = df[df['path'] == filename]

    print(rows[['source', 'type', 'method', 'race', 'gender']].head(20))